# DS Week 07 — Phishing Detection Under Unseen-Domain Shift
**Senior Data Scientist lab | 24 October 2026 | Security / URL-only tabular ML**

**Research question:** Does a model that performs well on seen domains maintain discrimination, calibration, and acceptable operational costs on *previously unseen* domains? We treat this as a **generalization and decision-science** experiment, not a leaderboard chase. The notebook explicitly distinguishes `REAL_UCI` from `SYNTHETIC_SMOKE`.

**Hypotheses:** H1: grouped-domain evaluation is more realistic than a random URL split; H2: domain-aware lexical features plus tree models outperform a naive linear baseline on validation PR-AUC; H3: post-hoc calibration improves probabilistic decision quality, but may not always improve ranking; H4: a small parameter-sharing ensemble can compete with classical baselines without large GPU memory.

**Business framing:** Analysts pay for false alarms; users incur harm from missed phishing. All monetary costs in this notebook are *illustrative sensitivity parameters*, not measured industry costs. We do not auto-block URLs based on these scores.

## 1. Data provenance, ethics and target
Source: [UCI PhiUSIIL, dataset 967](https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset), Prasad & Chandra (2024), **CC BY 4.0**. 235,795 rows, URL/page features; original label `0=phishing`, `1=legitimate`. We define `target = 1-label` to make `1=phishing`. Only the **URL string** is permitted as input; we do **not** use post-fetch HTML, FILENAME, domain identity, URLSimilarityIndex or TLDLegitimateProb as predictors. See `DATASET.md`.

**Access:** manually download the UCI ZIP and extract `PhiUSIIL_Phishing_URL_Dataset.csv` to `data/`. Dataset is not bundled. If absent, this notebook explicitly runs a reserved `.example` **synthetic smoke path**; its metrics are *not evidence of real-world phishing detection*.

**Ethical boundary:** never browse suspicious URLs from this notebook. Lexical feature extraction is offline and makes no HTTP requests.

In [ ]:
from pathlib import Path
import sys, json, time, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
from phishing_lab import (SEED, clean_frame, make_synthetic, featurize, group_splits,
                          score, grouped_bootstrap_ci, tracking_perturb)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import average_precision_score, ConfusionMatrixDisplay, PrecisionRecallDisplay

np.random.seed(SEED)
ART = ROOT/'artifacts'; ART.mkdir(exist_ok=True)
DATA = ROOT/'data'/'PhiUSIIL_Phishing_URL_Dataset.csv'
MODE = 'REAL_UCI' if DATA.is_file() else 'SYNTHETIC_SMOKE'
print('DATA MODE:', MODE, '| dataset:', DATA if DATA.is_file() else 'synthetic reserved .example URLs')

DATA MODE: SYNTHETIC_SMOKE | dataset: synthetic reserved .example URLs


## 2. Ingestion, deduplication and leakage audit
Rows with contradictory labels for an identical URL are discarded entirely, then identical URLs are deduplicated. The URL is the prediction-time information boundary. **Domain group** is split metadata only and never enters `X`. Host grouping uses a documented approximation to registrable domains; production validation must use a pinned Public Suffix List. Dataset timestamps are unavailable, so this is **not** a temporal generalization study.

In [ ]:
raw = pd.read_csv(DATA, usecols=['URL','label'], low_memory=False) if MODE == 'REAL_UCI' else make_synthetic(seed=SEED, domains=500, urls_per_domain=4)
df, cleaning = clean_frame(raw)
assert df['target'].isin([0,1]).all()
print('Raw rows:',len(raw),'clean:',len(df),'domain groups:',df.domain_group.nunique())
print('Cleaning audit:',cleaning)
print('Target prevalence:',round(float(df.target.mean()),4))
assert not df.URL.duplicated().any()

Raw rows: 2000 clean: 2000 domain groups: 500
Cleaning audit: {'conflicting_urls_removed': 0, 'clean_rows': 2000, 'unique_groups': 500}
Target prevalence: 0.432


## 3. EDA: imbalance, missingness, outliers, group concentration
We inspect class balance, group sizes, URL lengths and potential malformed URLs. Outliers are **not** automatically deleted: unusually long URLs and unusual characters may be the signal of interest. Missing values in extracted features would be imputed *within the training pipeline*, not globally.

In [ ]:
url_len = df.URL.str.len()
quality = pd.DataFrame({'item':['rows','phishing_prevalence','missing_urls','unique_domains','largest_domain_rows','url_length_p50','url_length_p99','url_length_max'],
'value':[len(df),float(df.target.mean()),int(df.URL.isna().sum()),df.domain_group.nunique(),int(df.domain_group.value_counts().max()),float(url_len.median()),float(url_len.quantile(.99)),int(url_len.max())]})
display(quality)
fig,ax=plt.subplots(1,2,figsize=(10,3.5))
df.target.value_counts().sort_index().plot.bar(ax=ax[0],title='Labels (1=phishing)')
url_len.clip(upper=url_len.quantile(.99)).hist(ax=ax[1],bins=30)
ax[1].set_title('URL length (clipped at p99 for display)')
plt.tight_layout(); plt.savefig(ART/f'{MODE.lower()}_eda.png',dpi=115);plt.show()

                  item     value
0                 rows  2000.000
1  phishing_prevalence     0.432
2         missing_urls     0.000
3       unique_domains   500.000
4  largest_domain_rows     4.000
5       url_length_p50    51.000
6       url_length_p99    69.000
7       url_length_max    72.000

<Figure size 1000x350 with 2 Axes>

## 4. Domain-aware, leakage-safe feature engineering
All 20 engineered features come solely from the URL available *before any web request*: hostname/path lengths, digit and punctuation ratios, Shannon character entropy, depth, query parameters, suspicious lexical tokens, punycode, IP-host indicator, HTTPS scheme and encoded-character count. No target encoding, training-domain lookup, page content, or post-incident information. These features capture obfuscation and impersonation hypotheses but are not causal evidence of phishing.

We store features as `float32`; extraction has deterministic behavior and no network dependency. No sampling of the real dataset is performed.

In [ ]:
t0=time.perf_counter()
X=featurize(df.URL)
y=df.target.to_numpy(dtype='int8')
assert len(X)==len(df) and X.dtypes.eq('float32').all()
assert not any(c in X.columns for c in ['label','target','Domain','URL'])
print('X:',X.shape,'RAM MiB:',round(X.memory_usage(deep=True).sum()/2**20,2),'feature seconds:',round(time.perf_counter()-t0,2))
display(pd.DataFrame({'missing':X.isna().sum(),'p99':X.quantile(.99),'max':X.max()}).sort_values('p99',ascending=False).head(12))

X: (2000, 20) RAM MiB: 0.15 feature seconds: 0.13


                  missing        p99        max
url_length              0  69.000000  72.000000
host_length             0  26.000000  26.000000
query_length            0  23.000000  23.000000
path_length             0  19.000000  19.000000
max_token_length        0  11.000000  11.000000
url_entropy             0   4.814244   4.892287
host_entropy            0   4.061482   4.061482
host_digits             0   4.000000   4.000000
suspicious_count        0   3.000000   3.000000
param_count             0   3.000000   3.000000
path_depth              0   2.000000   2.000000
subdomain_count         0   1.000000   1.000000

## 5. Validation protocol and contamination tests
Four **group-disjoint** partitions: train for fitting, validation for model selection, calibration for Platt/sigmoid calibration, and untouched test for final reporting. A random URL split is intentionally not used as the main metric because near-duplicate domains would inflate results. Split proportions are approximate since group sizes vary. We inspect class balance per partition; failures are explicit, not silently reseeded.

In [ ]:
splits=group_splits(df,SEED)
split_table=pd.DataFrame([{'partition':name,'n':len(ix),'groups':df.iloc[ix].domain_group.nunique(),'phishing_rate':float(y[ix].mean())} for name,ix in splits.items()])
display(split_table)
assert sum(map(len,splits.values()))==len(df)
assert len(set(df.iloc[splits['test']].domain_group)&set(df.iloc[splits['train']].domain_group))==0
assert all(np.isfinite(X.to_numpy().ravel()))

     partition     n  groups  phishing_rate
0        train  1196     299       0.428094
1   validation   300      75       0.426667
2  calibration   204      51       0.450980
3         test   300      75       0.440000

## 6. Baselines, bounded hyperparameters and selection criterion
Fit a standardized logistic model, histogram gradient boosting and ExtraTrees. The classical candidates use modest fixed configurations instead of an expensive search. In a follow-up, use a *nested group-aware* randomized search on training folds only; do not repeatedly optimize on the test set. **Select by validation PR-AUC**, appropriate when attack prevalence varies and positive alerts matter. Calibration is a separate operation after model selection.

In [ ]:
models={
 'logistic':make_pipeline(StandardScaler(),LogisticRegression(max_iter=600,class_weight='balanced',random_state=SEED)),
 'hist_gbdt':HistGradientBoostingClassifier(max_iter=100,max_leaf_nodes=15,learning_rate=.07,l2_regularization=2,random_state=SEED),
 'extra_trees':ExtraTreesClassifier(n_estimators=90,min_samples_leaf=4,max_features=.85,n_jobs=4,class_weight='balanced',random_state=SEED),
}
validation=[]
for name,model in models.items():
    start=time.perf_counter(); model.fit(X.iloc[splits['train']],y[splits['train']]);
    p=model.predict_proba(X.iloc[splits['validation']])[:,1]
    validation.append({'model':name,'validation_pr_auc':float(average_precision_score(y[splits['validation']],p)),'fit_seconds':round(time.perf_counter()-start,3)})
validation_df=pd.DataFrame(validation).sort_values('validation_pr_auc',ascending=False)
display(validation_df)
selected_name=validation_df.iloc[0]['model']; selected=models[selected_name]
print('Selected classical model:',selected_name)

         model  validation_pr_auc  fit_seconds
0     logistic           0.755997        0.027
1    hist_gbdt           0.667592        0.265
2  extra_trees           0.664421        0.318

Selected classical model: logistic


## 7. Research adaptation: lightweight parameter-sharing neural ensemble
**Paper:** Gorishniy, Kotelnikov & Babenko, *TabM*, ICLR 2025. The authors propose parameter-efficient tabular ensembling and report benchmark results on their datasets. Here we implement a **small BatchEnsemble-inspired shared-backbone adaptation**, not official TabM, with member-specific input adapters, shared layers, separate member losses, averaged probabilities, and validation early stopping. The classical selection protocol stays fixed; neural results are an independent research challenger. `torch` is optional; CPU fallback is automatic. The 4GB GPU is not required.

In [ ]:
neural_status={'status':'not_run'}
try:
    import torch
    from tabm_lite import fit_tabm_lite,tabm_predict
    neural_scaler=StandardScaler().fit(X.iloc[splits['train']])
    xx_train=neural_scaler.transform(X.iloc[splits['train']]).astype('float32')
    xx_val=neural_scaler.transform(X.iloc[splits['validation']]).astype('float32')
    neural_model,neural_status=fit_tabm_lite(xx_train,y[splits['train']],xx_val,y[splits['validation']],seed=SEED,epochs=18 if MODE=='REAL_UCI' else 8)
    neural_status['status']='completed'
    neural_status['validation_pr_auc']=float(average_precision_score(y[splits['validation']],tabm_predict(neural_model,xx_val)))
except ImportError as exc:
    neural_status={'status':'skipped_torch_unavailable','detail':str(exc)}
print(neural_status)

{'device': 'cpu', 'best_validation_pr_auc': 0.6667414572981604, 'epochs_ran': 8, 'ensemble_members': 4, 'status': 'completed', 'validation_pr_auc': 0.6667414572981604}


## 8. Held-out probability calibration
Fit a 1D sigmoid calibration model on the **calibration** partition using clipped raw-score logits. This avoids using test labels for calibration. Validation was used for selection, not calibration fitting. For model selection alternatives, isotonic calibration requires more calibration data and risks overfitting. We report before/after Brier score to avoid assuming calibration necessarily helps.

In [ ]:
def logit_features(p):
    q=np.clip(np.asarray(p),1e-6,1-1e-6)
    return np.log(q/(1-q)).reshape(-1,1)
cal_raw=selected.predict_proba(X.iloc[splits['calibration']])[:,1]
calibrator=LogisticRegression(max_iter=300,random_state=SEED).fit(logit_features(cal_raw),y[splits['calibration']])
test_raw=selected.predict_proba(X.iloc[splits['test']])[:,1]
test_p=calibrator.predict_proba(logit_features(test_raw))[:,1]
y_test=y[splits['test']]
print('Test Brier before:',round(score(y_test,test_raw)['brier'],4),'after:',round(score(y_test,test_p)['brier'],4))

Test Brier before: 0.1964 after: 0.1958


## 9. Test metrics, business costs and uncertainty
**Illustrative loss assumptions:** false positive = 5 units (analyst interruption), false negative = 25 units (missed attack). With calibrated probabilities and these costs, the Bayes threshold is `5/(5+25)=1/6`; costs must be validated against real deployment economics. Report threshold-independent PR-AUC/ROC-AUC, Brier/log loss, and thresholded operational outcomes. Grouped bootstrap resamples *domains*, not individual URLs, to better reflect dependence.

In [ ]:
C_FP,C_FN=5.,25.
threshold=C_FP/(C_FP+C_FN)
final_metrics=score(y_test,test_p,threshold=threshold,fp_cost=C_FP,fn_cost=C_FN)
ci=grouped_bootstrap_ci(y_test,test_p,df.iloc[splits['test']].domain_group.to_numpy(),metric='pr_auc',n_boot=120)
final_metrics['domain_bootstrap_pr_auc_95pct_ci']=ci
print('MODE',MODE,'threshold',threshold)
display(pd.DataFrame([final_metrics]).T.rename(columns={0:'value'}))
fig,ax=plt.subplots(figsize=(5.5,4))
PrecisionRecallDisplay.from_predictions(y_test,test_p,ax=ax)
plt.tight_layout();plt.savefig(ART/f'{MODE.lower()}_precision_recall.png',dpi=120);plt.show()

MODE SYNTHETIC_SMOKE threshold 0.16666666666666666


                                                                     value
n                                                                      300
prevalence                                                            0.44
pr_auc                                                            0.725543
roc_auc                                                           0.758207
brier                                                             0.195848
log_loss                                                          0.580443
precision                                                         0.440678
recall                                                            0.984848
f1                                                                0.608899
false_positives                                                        165
false_negatives                                                          2
cost_per_url                                                      2.916667
domain_bootstrap_pr_auc_9

<Figure size 550x400 with 1 Axes>

## 10. Error/slice analysis and interpretation
Slice the held-out domains by URL length (train-derived median) and HTTPS indicator. Interpret slices cautiously: slice sizes and prevalence differ, so compare confidence intervals before acting. Feature importance is measured via validation-set **permutation** of the selected model, not leakage-prone global feature-label correlations.

In [ ]:
length_cut=float(X.iloc[splits['train']]['url_length'].median())
slice_labels={'short_url':X.iloc[splits['test']]['url_length'].to_numpy()<=length_cut,
              'long_url':X.iloc[splits['test']]['url_length'].to_numpy()>length_cut,
              'https':X.iloc[splits['test']]['is_https'].to_numpy()==1,
              'non_https':X.iloc[splits['test']]['is_https'].to_numpy()==0}
slice_results=[]
for label,mask in slice_labels.items():
    if mask.sum()>=10 and len(set(y_test[mask]))==2:
        row=score(y_test[mask],test_p[mask],threshold=threshold)
        slice_results.append({'slice':label,**row})
display(pd.DataFrame(slice_results)[['slice','n','prevalence','pr_auc','recall','cost_per_url']])
importance=permutation_importance(selected,X.iloc[splits['validation']],y[splits['validation']],scoring='average_precision',n_repeats=3,random_state=SEED,n_jobs=2)
imp=pd.Series(importance.importances_mean,index=X.columns).sort_values(ascending=False)
display(imp.head(10).to_frame('validation_AP_drop'))
fig,ax=plt.subplots(figsize=(7,4));imp.head(10).sort_values().plot.barh(ax=ax);ax.set_title('Validation permutation importance (AP drop)');plt.tight_layout();plt.savefig(ART/f'{MODE.lower()}_feature_importance.png',dpi=120);plt.show()

       slice    n  prevalence    pr_auc   recall  cost_per_url
0  short_url  177    0.361582  0.663759  0.96875      3.418079
1   long_url  123    0.552846  0.787765  1.00000      2.195122
2      https  219    0.383562  0.583329  0.97619      3.242009
3  non_https   81    0.592593  0.911606  1.00000      2.037037

                  validation_AP_drop
suspicious_count            0.298088
host_length                 0.160009
max_token_length            0.116852
subdomain_count             0.110639
is_https                    0.060452
path_length                 0.031845
url_length                  0.028638
param_count                 0.018241
digit_ratio                 0.010398
special_ratio               0.010285

<Figure size 700x400 with 1 Axes>

## 11. Robustness and calibration diagnostics
We append an innocuous tracking query parameter without changing the hostname; predicted risk should ideally be stable under this benign modification. This is a *sensitivity test*, not a complete adversarial security audit. Also plot reliability bins and test different cost assumptions. If calibration worsens Brier score, do not hide the result.

In [ ]:
perturbed=featurize(df.iloc[splits['test']].URL.map(tracking_perturb))
perturbed_raw=selected.predict_proba(perturbed)[:,1]
perturbed_p=calibrator.predict_proba(logit_features(perturbed_raw))[:,1]
robustness={'mean_abs_probability_shift':float(np.mean(np.abs(test_p-perturbed_p))),
            'p95_abs_probability_shift':float(np.quantile(np.abs(test_p-perturbed_p),.95)),
            'decision_flip_rate':float(np.mean((test_p>=threshold)!=(perturbed_p>=threshold)))}
print('Benign query perturbation:',robustness)
from sklearn.calibration import calibration_curve
obs,pred=calibration_curve(y_test,test_p,n_bins=8,strategy='quantile')
fig,ax=plt.subplots(figsize=(5,4));ax.plot([0,1],[0,1],'--',label='ideal');ax.plot(pred,obs,'o-',label='calibrated');ax.set(xlabel='Mean predicted risk',ylabel='Observed phishing rate',title='Test reliability diagram');ax.legend();plt.tight_layout();plt.savefig(ART/f'{MODE.lower()}_reliability.png',dpi=120);plt.show()
cost_sensitivity=[]
for fn_cost in [10.,25.,50.,100.]:
    t=C_FP/(C_FP+fn_cost)
    cost_sensitivity.append({'FN_cost':fn_cost,'threshold':t,'cost_per_url':score(y_test,test_p,t,C_FP,fn_cost)['cost_per_url']})
display(pd.DataFrame(cost_sensitivity))

Benign query perturbation: {'mean_abs_probability_shift': 0.21909530183476733, 'p95_abs_probability_shift': 0.3116868659945396, 'decision_flip_rate': 0.46}


<Figure size 500x400 with 1 Axes>

   FN_cost  threshold  cost_per_url
0     10.0   0.333333      2.050000
1     25.0   0.166667      2.916667
2     50.0   0.090909      2.800000
3    100.0   0.047619      2.800000

## 12. Experiment comparison, conclusions, limitations and research follow-ups
The classical winner was selected on validation PR-AUC; the optional neural challenger is a research adaptation rather than an official TabM reproduction. **Synthetic smoke results do not support deployment claims.** The test split is domain-disjoint, but may still share hosting providers, registrars, phishing kits or URL templates. Without timestamps and external datasets, this cannot establish forward-time or cross-dataset robustness. A production follow-up should add a pinned PSL, prospective evaluation, out-of-distribution abstention, prevalence-shift recalibration, human review, cost elicitation, and fairness/false-blocking audits.

**Interview-ready trade-off:** group split prevents direct domain memorization; separate calibration partition prevents probability fitting on test; URL-only predictors meet the pre-click availability constraint; classical baselines remain strong and explainable; research adaptation tests efficiency without asserting the paper's benchmark results.

In [ ]:
comparison=validation_df.to_dict(orient='records')
if neural_status.get('status')=='completed':
    comparison.append({'model':'tabm_lite_research_challenger','validation_pr_auc':neural_status['validation_pr_auc'],'fit_seconds':None})
report={'run_mode':MODE,'seed':SEED,'rows':len(df),'cleaning':cleaning,
        'split_summary':split_table.to_dict(orient='records'),'model_selection':'classical validation PR-AUC',
        'selected_model':selected_name,'validation_comparison':comparison,'neural_research_adaptation':neural_status,
        'test_metrics':final_metrics,'robustness':robustness,'slice_metrics':slice_results,
        'cost_sensitivity':cost_sensitivity,'research_claim_boundary':'This is a BatchEnsemble-inspired adaptation, not official TabM.',
        'real_dataset_executed':MODE=='REAL_UCI'}
output=ART/f'{MODE.lower()}_metrics.json'
output.write_text(json.dumps(report,indent=2,default=float),encoding='utf-8')
validation_df.to_csv(ART/f'{MODE.lower()}_model_comparison.csv',index=False)
imp.to_csv(ART/f'{MODE.lower()}_permutation_importance.csv',header=['validation_AP_drop'])
print('Wrote:',output,'| real dataset executed:',report['real_dataset_executed'])
print('Conclusions:',('Real UCI results are available; validate externally before use.' if MODE=='REAL_UCI' else 'Synthetic smoke run only; download UCI dataset and rerun for empirical conclusions.'))

Wrote: /mnt/data/DS-Week-07-Phishing-Domain-Generalization/artifacts/synthetic_smoke_metrics.json | real dataset executed: False
Conclusions: Synthetic smoke run only; download UCI dataset and rerun for empirical conclusions.
